# MÁSTER BIG DATA & DATA ENGINEERING
## Programación Avanzada en Python — Trabajo Final

### ETAPA 1: Análisis exploratorio de multas de tráfico de Madrid (diciembre 2024)

## 0. Importación de librerías

Importamos todas las librerías necesarias al inicio del notebook, siguiendo el estilo PEP-8.

In [1]:
import io
import re
import urllib
from os import listdir

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

## 1. Descarga y carga de datos

Descargamos los datos de multas de diciembre de 2024 desde el portal de datos abiertos del Ayuntamiento de Madrid usando `requests`.

In [2]:
# URL del fichero CSV de multas de diciembre 2024
url = "https://datos.madrid.es/egob/catalogo/210104-395-multas-circulacion-detalle.csv"

# Realizamos la petición GET con timeout para evitar esperas indefinidas
response = requests.get(url, timeout=30)
response.raise_for_status()

# Mostramos la URL final tras la redirección
print(f"Descarga correcta. URL final tras redirección: {response.url}")

Descarga correcta. URL final tras redirección: https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/resource/210104-15-multas-circulacion-detalle-csv/download/210104-15-multas-circulacion-detalle-csv


Como puede observarse, el servidor redirige la URL original a la URL definitiva del fichero CSV.

Ahora leemos el contenido con `pandas.read_csv`, usando el separador `;` y la codificación `cp1252`. Leemos desde `response.content` para conservar los bytes originales y evitar problemas de caracteres mal decodificados.

In [3]:
# Leemos los bytes originales de la respuesta HTTP para controlar la codificación correctamente
content = io.BytesIO(response.content)

# Leemos el CSV con separador ; y codificación cp1252
multas = pd.read_csv(content, sep=';', encoding='cp1252')

# Mostramos las primeras filas para verificar la carga
multas.head()

,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
0,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA. ...,,,,
1,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",,,,
2,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN...,,,,
3,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA. ...,,,,
4,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",,,,


## 2. Análisis exploratorio

### 2.1 Estructura general del DataFrame (`df.info()`)

In [4]:
# Información general del DataFrame
multas.info()

<class 'pandas.DataFrame'>
RangeIndex: 249801 entries, 0 to 249800
Data columns (total 14 columns):
 #   Column                                                                                                                                                  Non-Null Count   Dtype  
---  ------                                                                                                                                                  --------------   -----  
 0   CALIFICACION                                                                                                                                            249801 non-null  str    
 1   LUGAR                                                                                                                                                   249801 non-null  str    
 2   MES                                                                                                                                                     249801 non-null

**Comentarios sobre el resultado de `info()`:**

- El DataFrame tiene 14 columnas y 249801 filas.
- En esta versión de pandas, las columnas de texto aparecen con tipo `str`; las numéricas visibles desde el principio son `MES`, `ANIO` y `PUNTOS` (`int64`), además de `HORA` e `IMP_BOL` (`float64`).
- `HORA` está en formato decimal aparente (p.ej. `20.23` representa 20h 23min), por eso se usará después para construir una columna temporal más expresiva.
- Las columnas `VEL_LIMITE`, `VEL_CIRCULA`, `COORDENADA-X` y `COORDENADA-Y` todavía aparecen como texto porque contienen espacios en blanco y valores vacíos que más adelante convertiremos a numéricos.
- En este punto los recuentos no nulos son completos porque los campos vacíos todavía están como cadenas vacías; al convertir esas columnas con `pd.to_numeric(errors='coerce')`, esos huecos pasarán a `NaN`.

In [5]:
# Número de filas y columnas
print(f"Número de filas: {multas.shape[0]}")
print(f"Número de columnas: {multas.shape[1]}")

Número de filas: 249801
Número de columnas: 14


In [6]:
# Tipos de datos de cada columna
print("Tipos de datos:")
print(multas.dtypes)

Tipos de datos:
CALIFICACION                                                                                                                                                  str
LUGAR                                                                                                                                                         str
MES                                                                                                                                                         int64
ANIO                                                                                                                                                        int64
HORA                                                                                                                                                      float64
IMP_BOL                                                                                                                                                   float64
DESCUENTO   

In [7]:
# Valores no nulos por columna
print("Valores no nulos por columna:")
print(multas.notnull().sum())

Valores no nulos por columna:
CALIFICACION                                                                                                                                              249801
LUGAR                                                                                                                                                     249801
MES                                                                                                                                                       249801
ANIO                                                                                                                                                      249801
HORA                                                                                                                                                      249801
IMP_BOL                                                                                                                                                   249801
DESC

## 3. Limpieza de datos

### 3a. Columnas categóricas: eliminar espacios en blanco

In [8]:
# a) Columna CALIFICACION: número de valores únicos ANTES de limpiar
print("Valores únicos en CALIFICACION (antes):", multas['CALIFICACION'].nunique())
print(multas['CALIFICACION'].unique())

# Eliminar espacios en blanco al inicio y final
multas['CALIFICACION'] = multas['CALIFICACION'].str.strip()

# Número de valores únicos DESPUÉS de limpiar
print("\nValores únicos en CALIFICACION (después):", multas['CALIFICACION'].nunique())
print(multas['CALIFICACION'].unique())

Valores únicos en CALIFICACION (antes): 3
<StringArray>
['LEVE      ', 'GRAVE     ', 'MUY GRAVE ']
Length: 3, dtype: str

Valores únicos en CALIFICACION (después): 3
<StringArray>
['LEVE', 'GRAVE', 'MUY GRAVE']
Length: 3, dtype: str


In [9]:
# b) Columnas DESCUENTO, HECHO-BOL y DENUNCIANTE
for col in ['DESCUENTO', 'HECHO-BOL', 'DENUNCIANTE']:
    print(f"Valores únicos en {col} (antes): {multas[col].nunique()}")
    multas[col] = multas[col].str.strip()
    print(f"Valores únicos en {col} (después): {multas[col].nunique()}")
    print()

Valores únicos en DESCUENTO (antes): 1
Valores únicos en DESCUENTO (después): 1

Valores únicos en HECHO-BOL (antes): 345
Valores únicos en HECHO-BOL (después): 345

Valores únicos en DENUNCIANTE (antes): 6
Valores únicos en DENUNCIANTE (después): 6



### 3b. Nombres de columnas: detectar y eliminar espacios

In [10]:
# c) Mostrar nombres de columnas actuales
print("Nombres de columnas (con posibles espacios):")
for col in multas.columns:
    print(repr(col))  # repr() muestra los espacios invisibles

Nombres de columnas (con posibles espacios):
'CALIFICACION'
'LUGAR'
'MES'
'ANIO'
'HORA'
'IMP_BOL'
'DESCUENTO'
' PUNTOS'
'DENUNCIANTE'
'HECHO-BOL'
'VEL_LIMITE'
'VEL_CIRCULA '
'COORDENADA-X'
'COORDENADA-Y                                                                                                                                          '


In [11]:
# Detectar columnas con espacios al final
cols_con_espacios = [col for col in multas.columns if col != col.strip()]
print(f"Columnas con espacios: {cols_con_espacios}")

# Mostrar la columna VEL_CIRCULA (puede tener espacio al final)
print("\nColumna de velocidad de circulación:", repr([c for c in multas.columns if 'VEL_CIRCULA' in c][0]))

# Mostrar la columna COORDENADA-Y (puede tener espacios al final)
print("Columna COORDENADA-Y:", repr([c for c in multas.columns if 'COORDENADA' in c and 'Y' in c][0]))

Columnas con espacios: [' PUNTOS', 'VEL_CIRCULA ', 'COORDENADA-Y                                                                                                                                          ']

Columna de velocidad de circulación: 'VEL_CIRCULA '
Columna COORDENADA-Y: 'COORDENADA-Y                                                                                                                                          '


In [12]:
# Serie de datos correspondiente a COORDENADA-Y (antes de renombrar)
col_coord_y = [c for c in multas.columns if 'COORDENADA' in c and 'Y' in c][0]
print(f"Serie COORDENADA-Y (columna: {repr(col_coord_y)}):")
multas[col_coord_y].head(10)

Serie COORDENADA-Y (columna: 'COORDENADA-Y                                                                                                                                          '):


0               
1               
2               
3               
4               
5               
6               
7               
8               
9               
Name: COORDENADA-Y                                                                                                                                          , dtype: str

In [13]:
# Renombrar columnas: eliminar espacios al inicio y final de todos los nombres
multas.rename(columns=lambda x: x.strip(), inplace=True)

print("Nombres de columnas (tras limpiar):")
for col in multas.columns:
    print(repr(col))

Nombres de columnas (tras limpiar):
'CALIFICACION'
'LUGAR'
'MES'
'ANIO'
'HORA'
'IMP_BOL'
'DESCUENTO'
'PUNTOS'
'DENUNCIANTE'
'HECHO-BOL'
'VEL_LIMITE'
'VEL_CIRCULA'
'COORDENADA-X'
'COORDENADA-Y'


### 3c. Columnas de velocidad: conversión a numérico

In [14]:
# d) Tipo actual de las columnas de velocidad
print("Tipo actual de VEL_LIMITE:", multas['VEL_LIMITE'].dtype)
print("Tipo actual de VEL_CIRCULA:", multas['VEL_CIRCULA'].dtype)

# Valores únicos (puede haber strings vacíos o con espacios)
print("\nValores únicos en VEL_LIMITE:", multas['VEL_LIMITE'].nunique())
print(multas['VEL_LIMITE'].unique()[:10])

print("\nValores únicos en VEL_CIRCULA:", multas['VEL_CIRCULA'].nunique())
print(multas['VEL_CIRCULA'].unique()[:10])

Tipo actual de VEL_LIMITE: str
Tipo actual de VEL_CIRCULA: str

Valores únicos en VEL_LIMITE: 7
<StringArray>
['   ', ' 50', ' 60', ' 40', ' 30', ' 90', ' 70']
Length: 7, dtype: str

Valores únicos en VEL_CIRCULA: 101
<StringArray>
['   ', ' 74', ' 79', '100', ' 73', '103', ' 96', ' 91', ' 77', ' 95']
Length: 10, dtype: str


In [15]:
# Convertir a numérico: errors='coerce' convierte los valores no numéricos a NaN
multas['VEL_LIMITE'] = pd.to_numeric(multas['VEL_LIMITE'], errors='coerce')
multas['VEL_CIRCULA'] = pd.to_numeric(multas['VEL_CIRCULA'], errors='coerce')

print("Tipo de VEL_LIMITE tras conversión:", multas['VEL_LIMITE'].dtype)
print("Tipo de VEL_CIRCULA tras conversión:", multas['VEL_CIRCULA'].dtype)

Tipo de VEL_LIMITE tras conversión: float64
Tipo de VEL_CIRCULA tras conversión: float64


### 3d. Coordenadas: conversión a numérico

In [16]:
# e) Tipo actual de las columnas de coordenadas
print("Tipo actual de COORDENADA-X:", multas['COORDENADA-X'].dtype)
print("Tipo actual de COORDENADA-Y:", multas['COORDENADA-Y'].dtype)

# Convertir a numérico
multas['COORDENADA-X'] = pd.to_numeric(multas['COORDENADA-X'], errors='coerce')
multas['COORDENADA-Y'] = pd.to_numeric(multas['COORDENADA-Y'], errors='coerce')

print("\nTipo de COORDENADA-X tras conversión:", multas['COORDENADA-X'].dtype)
print("Tipo de COORDENADA-Y tras conversión:", multas['COORDENADA-Y'].dtype)

Tipo actual de COORDENADA-X: str
Tipo actual de COORDENADA-Y: str

Tipo de COORDENADA-X tras conversión: float64
Tipo de COORDENADA-Y tras conversión: float64


## 4. Crear columna fecha

### a) Crear la columna `fecha` combinando ANIO, MES y HORA

La columna HORA está en formato decimal (p.ej. `20.23` = 20h 23min). Usamos día fijo = 1.

In [17]:
# Extraer la parte entera (hora) y la parte decimal (minutos)
hora_decimal = pd.to_numeric(multas['HORA'], errors='coerce').fillna(0)
hora_int = hora_decimal.astype(int)
minuto_int = ((hora_decimal - hora_int) * 100).round().astype(int)

# Crear la columna fecha usando pd.to_datetime
multas['fecha'] = pd.to_datetime(
    {
        'year': multas['ANIO'],
        'month': multas['MES'],
        'day': 1,
        'hour': hora_int,
        'minute': minuto_int,
    },
    errors='coerce'
)

print("Tipo de la columna fecha:", multas['fecha'].dtype)
print(multas[['MES', 'ANIO', 'HORA', 'fecha']].head())

Tipo de la columna fecha: datetime64[us]
   MES  ANIO   HORA               fecha
0   12  2024  20.23 2024-12-01 20:23:00
1   12  2024  20.27 2024-12-01 20:27:00
2   12  2024  20.45 2024-12-01 20:45:00
3   12  2024  16.30 2024-12-01 16:30:00
4   12  2024  16.50 2024-12-01 16:50:00


### b) Establecer `fecha` como índice del DataFrame

In [18]:
# Establecer la columna fecha como índice
multas.set_index('fecha', inplace=True)

print("Índice del DataFrame:", multas.index.dtype)
multas.head(3)

Índice del DataFrame: datetime64[us]


,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
fecha,,,,,,,,,,,,,,
2024-12-01 20:23:00,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.,NaN,NaN,NaN,NaN
2024-12-01 20:27:00,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",NaN,NaN,NaN,NaN
2024-12-01 20:45:00,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN...,NaN,NaN,NaN,NaN


### c) Eliminar la columna `fecha` auxiliar

Como ya hemos establecido `fecha` como índice, si existiera como columna la eliminaríamos. En pandas, `set_index` ya la elimina automáticamente, pero por completitud:

In [19]:
# Eliminar la columna 'fecha' si aún existiera como columna (por seguridad)
if 'fecha' in multas.columns:
    multas.drop(columns=['fecha'], inplace=True)

print("Columnas finales:")
print(list(multas.columns))
print("\nÍndice:")
print(multas.index[:5])

Columnas finales:
['CALIFICACION', 'LUGAR', 'MES', 'ANIO', 'HORA', 'IMP_BOL', 'DESCUENTO', 'PUNTOS', 'DENUNCIANTE', 'HECHO-BOL', 'VEL_LIMITE', 'VEL_CIRCULA', 'COORDENADA-X', 'COORDENADA-Y']

Índice:
DatetimeIndex(['2024-12-01 20:23:00', '2024-12-01 20:27:00',
               '2024-12-01 20:45:00', '2024-12-01 16:30:00',
               '2024-12-01 16:50:00'],
              dtype='datetime64[us]', name='fecha', freq=None)


## 5. Verificación del resultado final

In [20]:
# Resumen final del DataFrame limpio
print("=" * 60)
print("RESUMEN FINAL DEL DATAFRAME LIMPIO")
print("=" * 60)
print(f"Filas: {multas.shape[0]}")
print(f"Columnas: {multas.shape[1]}")
print()
multas.info()
print()
multas.head()

RESUMEN FINAL DEL DATAFRAME LIMPIO
Filas: 249801
Columnas: 14

<class 'pandas.DataFrame'>
DatetimeIndex: 249801 entries, 2024-12-01 20:23:00 to 2024-12-01 12:51:00
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   CALIFICACION  249801 non-null  str    
 1   LUGAR         249801 non-null  str    
 2   MES           249801 non-null  int64  
 3   ANIO          249801 non-null  int64  
 4   HORA          249801 non-null  float64
 5   IMP_BOL       249801 non-null  float64
 6   DESCUENTO     249801 non-null  str    
 7   PUNTOS        249801 non-null  int64  
 8   DENUNCIANTE   249801 non-null  str    
 9   HECHO-BOL     249801 non-null  str    
 10  VEL_LIMITE    22182 non-null   float64
 11  VEL_CIRCULA   22182 non-null   float64
 12  COORDENADA-X  123882 non-null  float64
 13  COORDENADA-Y  123882 non-null  float64
dtypes: float64(6), int64(3), str(5)
memory usage: 28.6 MB



,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
fecha,,,,,,,,,,,,,,
2024-12-01 20:23:00,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.,NaN,NaN,NaN,NaN
2024-12-01 20:27:00,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",NaN,NaN,NaN,NaN
2024-12-01 20:45:00,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN...,NaN,NaN,NaN,NaN
2024-12-01 16:30:00,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.,NaN,NaN,NaN,NaN
2024-12-01 16:50:00,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",NaN,NaN,NaN,NaN
